# Búsqueda local y optimización

Los algoritmos de [búsqueda](search.ipynb) devuelven un **camino**: la secuencia de acciones desde el inicio hasta el objetivo. En muchos problemas el camino no importa, solo el estado final: un horario, el diseño de un circuito, las posiciones de las reinas en un tablero. La **búsqueda local** parte de un candidato completo, por malo que sea, y lo mejora paso a paso, guiada por una **función objetivo** que puntúa cada candidato.

Solo guarda en memoria el candidato actual, nunca un árbol de caminos, así que funciona en espacios de estados demasiado grandes para recorrerlos sistemáticamente. El precio: se puede atascar, y no puede demostrar que no exista una solución mejor.

## El problema de las ocho reinas

Coloca ocho reinas en un tablero de ajedrez de forma que ninguna ataque a otra: ninguna pareja en la misma fila, columna o diagonal. Con una reina por columna, un candidato es una lista de ocho números de fila, uno por columna, y hay 8⁸ = 16.777.216 candidatos. El objetivo a minimizar es el número de parejas de reinas que se atacan; una solución tiene 0.

In [ ]:
SIZE = 8


def attacking_pairs(rows):
    pairs = 0
    for column_a in range(SIZE):
        for column_b in range(column_a + 1, SIZE):
            same_row = rows[column_a] == rows[column_b]
            same_diagonal = abs(rows[column_a] - rows[column_b]) == column_b - column_a
            pairs += same_row or same_diagonal
    return pairs


print(attacking_pairs([0, 1, 2, 3, 4, 5, 6, 7]), "attacking pairs with all the queens on one diagonal")

28 attacking pairs with all the queens on one diagonal


Los **vecinos** de un candidato son los candidatos que se diferencian en un movimiento: una reina movida a otra fila de su columna. Cada candidato tiene 8 × 7 = 56 vecinos.

In [ ]:
def neighbours(rows):
    for column in range(SIZE):
        for row in range(SIZE):
            if row != rows[column]:
                yield rows[:column] + [row] + rows[column + 1:]

## Ascensión de colinas

La **ascensión de colinas** (*hill climbing*), aquí como *descenso más pronunciado* porque el objetivo es un coste, se mueve al mejor vecino mientras sea mejor que el candidato actual, y se detiene cuando ninguno lo es. Donde se detiene es un **mínimo local**: ningún movimiento individual lo mejora, pero puede que no sea una solución.

In [ ]:
def hill_climb(rows):
    steps = 0
    while True:
        best = min(neighbours(rows), key=attacking_pairs)
        if attacking_pairs(best) >= attacking_pairs(rows):
            return rows, steps
        rows, steps = best, steps + 1

Mil tableros iniciales aleatorios, con una semilla fija para que los números sean los mismos en cada ejecución:

In [ ]:
import random

random.seed(0)
starts = [[random.randrange(SIZE) for _ in range(SIZE)] for _ in range(1000)]
results = [hill_climb(start) for start in starts]
solved = sum(attacking_pairs(rows) == 0 for rows, _ in results)
average_steps = sum(steps for _, steps in results) / len(results)
print(f"Solved {solved} of {len(starts)} boards, in {average_steps:.1f} steps on average")

Solved 161 of 1000 boards, in 3.3 steps on average


La ascensión de colinas es rápida, unos pocos pasos por tablero, y falla la mayoría de las veces: la mayoría de los tableros iniciales llevan a un mínimo local con una pareja de reinas que todavía se atacan. El libro de texto de Russell y Norvig informa del mismo comportamiento, alrededor del 14 % de tableros resueltos, con una regla algo distinta para los empates.

## Reinicios aleatorios

El remedio más sencillo: cuando la ascensión de colinas se atasca, se vuelve a empezar desde un tablero aleatorio nuevo, hasta que una ejecución llega a una solución.

In [ ]:
random.seed(1)
restarts = 0
while True:
    restarts += 1
    rows, _ = hill_climb([random.randrange(SIZE) for _ in range(SIZE)])
    if attacking_pairs(rows) == 0:
        break
print(f"Solved after {restarts} starts: {rows}")

Solved after 4 starts: [5, 1, 6, 0, 2, 4, 7, 3]


## Recocido simulado

El **recocido simulado** (*simulated annealing*; Kirkpatrick, Gelatt y Vecchi, 1983) permite a la búsqueda escapar de los mínimos locales aceptando a veces un movimiento **peor**. Elige un vecino al azar; uno mejor siempre se acepta, y uno peor con probabilidad e^(−Δ/T), donde Δ es cuánto peor es y *T* una **temperatura** que baja paso a paso. Al principio la búsqueda vaga casi libremente; a medida que se enfría, se comporta cada vez más como la ascensión de colinas. El nombre viene de la metalurgia, donde un metal enfriado lentamente se asienta en una estructura regular de baja energía.

In [ ]:
import math


def simulated_annealing(rows, temperature=2.0, cooling=0.995, max_steps=20_000):
    cost = attacking_pairs(rows)
    for step in range(max_steps):
        if cost == 0:
            return rows, step
        column, row = random.randrange(SIZE), random.randrange(SIZE)
        candidate = rows[:column] + [row] + rows[column + 1:]
        change = attacking_pairs(candidate) - cost
        if change <= 0 or random.random() < math.exp(-change / temperature):
            rows, cost = candidate, cost + change
        temperature = max(temperature * cooling, 1e-3)
    return rows, max_steps

In [ ]:
random.seed(0)
annealing_results = [simulated_annealing(start) for start in starts[:200]]
solved = sum(attacking_pairs(rows) == 0 for rows, _ in annealing_results)
print(f"Solved {solved} of 200 boards")

Solved 192 of 200 boards


Con los mismos tableros iniciales, aceptar algunos movimientos peores convierte la mayoría de los fallos en soluciones, a cambio de muchos más pasos, más baratos: cada uno mira un vecino aleatorio en lugar de los 56.

## Algoritmos genéticos

Un **algoritmo genético** (John Holland, 1975) mantiene toda una **población** de candidatos e imita la selección natural. En cada generación, la mitad más apta sobrevive como progenitores; cada hijo toma la primera parte de un progenitor y el resto de otro (**cruce**), y a veces un cambio aleatorio (**mutación**). El cruce puede combinar partes buenas encontradas por candidatos distintos, algo que una búsqueda local individual no puede hacer.

In [ ]:
def genetic_algorithm(population_size=100, mutation_rate=0.2, max_generations=1000):
    population = [[random.randrange(SIZE) for _ in range(SIZE)] for _ in range(population_size)]
    for generation in range(max_generations):
        population.sort(key=attacking_pairs)
        if attacking_pairs(population[0]) == 0:
            return population[0], generation
        parents = population[: population_size // 2]
        children = []
        while len(children) < population_size:
            mother, father = random.sample(parents, 2)
            cut = random.randrange(1, SIZE)
            child = mother[:cut] + father[cut:]
            if random.random() < mutation_rate:
                child[random.randrange(SIZE)] = random.randrange(SIZE)
            children.append(child)
        population = children
    return population[0], max_generations

In [ ]:
random.seed(0)
runs = [genetic_algorithm() for _ in range(20)]
generations = sorted(generation for _, generation in runs)
print("Solved", sum(attacking_pairs(best) == 0 for best, _ in runs), "of 20 runs")
print("Generations needed:", generations)

Solved 19 of 20 runs
Generations needed: [6, 9, 10, 12, 12, 13, 13, 13, 14, 14, 16, 23, 33, 34, 35, 38, 43, 451, 471, 1000]


La mayoría de las ejecuciones encuentran una solución en unas pocas decenas de generaciones; unas pocas vagan durante cientos, y una llegó al límite de 1.000 sin encontrarla. Los algoritmos genéticos necesitan ajustes, tamaño de la población, tasa de mutación, cómo se eligen los progenitores, y ninguna configuración funciona mejor para todos los problemas.

## De la búsqueda local al aprendizaje automático

Entrenar un modelo de aprendizaje automático también es búsqueda local, sobre un objetivo continuo. El candidato son los parámetros del modelo, el objetivo es el error con los datos de entrenamiento, y el **descenso del gradiente** se mueve a un vecino mejor en la dirección en la que, según el gradiente, el error baja más rápido ([cómo aprende una red neuronal](../../background/neural_networks.md#cómo-aprende)). Surgen las mismas preguntas: qué tamaño de paso, y cómo evitar atascarse. La idea del recocido simulado de un ruido que ayuda a escapar de las regiones malas tiene un primo en la aleatoriedad del descenso del gradiente estocástico.